# Bootstrap & Preflight — 10-Deployment_Drift.ipynb

**What this does**
- Verifies expected upstream notebooks have been executed and essential artefact folders exist.
- Prints clear guidance to run prerequisites if required files/folders are missing.

**Upstream prerequisites (recommended order)**
- `01-Setup_Preflight.ipynb`
- `04-Baseline_and_BO.ipynb`

**Checks performed**
- Confirms `DATA_PATH` exists (from Section 0.1).
- Ensures `staging/` and `out/` directories exist when required downstream.
- Provides actionable instructions when a check fails.


In [ ]:
# ======================================================
# Bootstrap & Preflight — 10-Deployment_Drift.ipynb
#   • Validates prerequisites and artefact folders
#   • Prints guidance if prerequisites are missing
# ======================================================
print(">>> Bootstrap & Preflight — 10-Deployment_Drift.ipynb")
required = ['01-Setup_Preflight.ipynb', '04-Baseline_and_BO.ipynb']
print("[bootstrap] Recommended upstream notebooks:", required)

# Check DATA_PATH existence if declared
if 'DATA_PATH' in globals():
    from pathlib import Path as _P
    dp = _P(DATA_PATH)
    if not dp.exists():
        print(f"[bootstrap][warn] DATA_PATH not found: {dp} — please verify in 01-Setup_Preflight (Section 0.1).")

# Check staging/out directories
from pathlib import Path as _P
if 'STAGE_ROOT' in globals():
    sr = _P(STAGE_ROOT); 
    if not sr.exists():
        print(f"[bootstrap][warn] STAGE_ROOT does not exist: {sr}. Run 01-Setup_Preflight end-to-end first.")
if 'OUT_ROOT' in globals():
    oroot = _P(OUT_ROOT);
    if not oroot.exists():
        print(f"[bootstrap][warn] OUT_ROOT does not exist: {oroot}. It will be created as needed, but prior steps may be required.")

# Feature artefacts helpful for downstream
from pathlib import Path as _P
feat_dir = _P('staging') / 'feat'
if not feat_dir.exists():
    print("[bootstrap][hint] 'staging/feat' not found — this notebook can generate it (Sections 3.3/3.4), or run 03-Feature_Selection first.")
else:
    mi_file = feat_dir / 'mi_series.csv'
    if not mi_file.exists():
        print("[bootstrap][hint] MI series not found at 'staging/feat/mi_series.csv' — run Section 3.3 to generate.")

print("[bootstrap] Preflight checks complete. Proceed with this notebook if no critical warnings above.")


# Section 10 — Preprocessor definition (structured; excludes payload)

In [ ]:
# =====================================================
# Section 10 — Preprocessor definition (structured; excludes payload)
# =====================================================
print(">>> Section 10: start")
try:
    _res_var = res if 'res' in locals() else results if 'results' in locals() else []
    # Prefer existing proba/pred; else derive via estimator if available
    if 'proba' in locals():
        _scores = proba
        _pred = (proba >= 0.5).astype(int)
    elif 'clf' in locals():
        _Xval = X_va_t if 'X_va_t' in locals() else X_val if 'X_val' in locals() else X_va if 'X_va' in locals() else None
        _scores, _pred = get_scores_and_pred(clf, _Xval)
    elif 'model' in locals():
        _Xval = X_va_t if 'X_va_t' in locals() else X_val if 'X_val' in locals() else X_va if 'X_va' in locals() else None
        _scores, _pred = get_scores_and_pred(model, _Xval)
    else:
        raise RuntimeError('no proba/estimator available for metrics hook')
    _fold = fold if 'fold' in locals() else 1
    _res_rec = fold_report("7.1", _fold, y_va, _scores, _pred)
    # Keep results list in 'res'
    res = res if 'res' in locals() else []
    res.append(_res_rec)
except Exception as _e:
    print(f"[warn] metrics hook skipped: {_e}")

# --- metrics summary (standard) ---
try:
    summary_report("7.1", res if 'res' in locals() else results if 'results' in locals() else [])
except Exception as _e:
    print(f"[warn] summary skipped: {_e}")

## Section 10.1 — Preprocessor definition (structured; excludes payload)

In [ ]:
# =====================================================
# Section 10 — Preprocessor definition (structured; excludes payload)
# =====================================================
print(">>> Section 10: start")